In [42]:
import warnings
warnings.filterwarnings("ignore")

# Unreleased feats

In the previous notebooks, we looked:
- How v2 works
- Its different layers
- Basic flow (using High level API or by low level 3 layered pipeline)'

In this notebook we will look at some unreleased feats of v2 some of which are still WiP

## Note
Please use the colab notebook for this, that will make the installation of different branches of the repo more fast and easily accessible

## Installation
Lets first install the `pytorch-forecasting` present on the main

In [1]:
!pip install git+https://github.com/sktime/pytorch-forecasting.git

  Cloning https://github.com/sktime/pytorch-forecasting.git to /tmp/pip-req-build-z9nu9erf
  Running command git clone --filter=blob:none --quiet https://github.com/sktime/pytorch-forecasting.git /tmp/pip-req-build-z9nu9erf
  Resolved https://github.com/sktime/pytorch-forecasting.git to commit 8918eac8cd4384cc9ef6c913bfcb08fbee9dcdc3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 44.0 MB/s eta 0:00:00
  Created wheel for pytorch-forecasting: filename=pytorch_forecasting-1.8.0-py3-none-any.whl size=438531 sha256=dd88a6b5d1a0b

Some unreleased feats:
- scalers and normalizer support in `EncoderDecoderTimeSeriesDataModule`
- Few more models (like `DecoderMLP`)
- In one of the branches: `save` and `load` interface for v2

### Scalers and normalizer support
- `EncoderDecoderTimeSeriesDataModule` is the only one that supports it, still need to add preprocessing to `tslib` DM
- Can use the normalizers provided by `pytorch_forecasting`, see `pytorch_forecasting.encoders` in the [docs](https://pytorch-forecasting.readthedocs.io/en/stable/data.html#details).
- Also supports `sklearn` scalers like `RobustScaler`, `StandardScaler` etc

For scalers support,
 - by default,  no scaling,
 - you can pass the scalers explicitly to the classes

Supported scalers
 * **PyTorch Forecasting Normalizers**:

     * `pytorch_forecasting.data.encoders.TorchNormalizer`
     * `pytorch_forecasting.data.encoders.GroupNormalizer`
     * `pytorch_forecasting.data.encoders.EncoderNormalizer`

 * **Scikit-Learn Scalers**:

     * ``StandardScaler``
     * ``RobustScaler``
     * ``MinMaxScaler``
     * ``MaxAbsScaler``


In [20]:
from pytorch_forecasting.data.data_module import EncoderDecoderTimeSeriesDataModule
from pytorch_forecasting.data.timeseries import TimeSeries

In [74]:
import numpy as np
import pandas as pd
def load_toydata(num_series, seq_length):
    data_list = []
    for i in range(num_series):
        x = np.arange(seq_length)
        level = 10 ** (i % 3)
        y = level * np.sin(x / 5.0) + np.random.normal(scale=0.1, size=seq_length)
        category = i % 5
        static_value = np.random.rand()
        for t in range(seq_length - 1):
            data_list.append(
                {
                    "series_id": i,
                    "time_idx": t,
                    "x": y[t],
                    "y": y[t + 1],
                    "category": category,
                    "future_known_feature": np.cos(t / 10),
                    "static_feature": static_value,
                    "static_feature_cat": i % 3,
                }
            )
    data_df = pd.DataFrame(data_list)
    return data_df

In [75]:
num_series = 100  # Number of individual time series to generate
seq_length = 50  # Length of each time series
data_df = load_toydata(num_series, seq_length)
data_df.head()

,series_id,time_idx,x,y,category,future_known_feature,static_feature,static_feature_cat
0,0,0,-0.004745,0.181007,0,1.000000,0.935732,0
1,0,1,0.181007,0.375809,0,0.995004,0.935732,0
2,0,2,0.375809,0.619213,0,0.980067,0.935732,0
3,0,3,0.619213,0.695514,0,0.955336,0.935732,0
4,0,4,0.695514,1.054218,0,0.921061,0.935732,0


In [76]:
# create `TimeSeries` dataset that returns the raw data in terms of tensors
dataset = TimeSeries(
    data=data_df,
    time="time_idx",
    target="y",
    group=["series_id"],
    num=["x", "future_known_feature", "static_feature"],
    cat=["category", "static_feature_cat"],
    known=["future_known_feature"],
    unknown=["x", "category"],
    static=["static_feature", "static_feature_cat"],
)

To make show the scalers work, we will skip `pkg` class usage for - although, you can simply pass the scalers to the configs of datamodule in case of using `pkg` class and the pipeline will initialise nad use the scalers

```python
datamodule_cfg = dict(
    max_encoder_length=30,
    max_prediction_length=1,
    batch_size=32,
    categorical_encoders={
        "category": NaNLabelEncoder(add_nan=True),
        "static_feature_cat": NaNLabelEncoder(add_nan=True),
    },
    scalers={
        "x": StandardScaler(),
        "future_known_feature": StandardScaler(),
        "static_feature": StandardScaler(),
    },
    target_normalizer=TorchNormalizer(),
)
```


##### Default is no scaling

In [26]:
# scalers=None → identity pass-through, raw values untouched
dm_raw = EncoderDecoderTimeSeriesDataModule(
    time_series_dataset=dataset,
    max_encoder_length=30,
    max_prediction_length=6,
)

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:157: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


##### Using scalers

In [40]:
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from pytorch_forecasting.data.encoders import GroupNormalizer, EncoderNormalizer, TorchNormalizer

dm_scaled = EncoderDecoderTimeSeriesDataModule(
    time_series_dataset=dataset,
    max_encoder_length=30,
    max_prediction_length=6,
    scalers={
        "x": StandardScaler(),
        "future_known_feature": MinMaxScaler(),
        "static_feature": EncoderNormalizer(),
    },
)

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:157: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


Lets compare both of the cases:
- Take the first batch from both dm's
- Plot line by line to compare both the dm's

In [28]:
import torch
def first_batch(dm, seed=42):
    torch.manual_seed(seed)      # split uses randperm — seed for a like-for-like compare
    dm.setup(stage="fit")
    return next(iter(dm.train_dataloader()))


In [30]:
x_raw, _    = first_batch(dm_raw)
x_scaled, _ = first_batch(dm_scaled)
cont_names = [
    dm_scaled.time_series_metadata["cols"]["x"][i]
    for i in dm_scaled.continuous_indices
]

print(f"{'feature':<24}{'raw mean':>10}{'raw std':>10} | {'scaled mean':>12}{'scaled std':>11}")
for i, name in enumerate(cont_names):
    r = x_raw["encoder_cont"][:, :, i]
    s = x_scaled["encoder_cont"][:, :, i]
    print(f"{name:<24}{r.mean():>10.3f}{r.std():>10.3f} | {s.mean():>12.3f}{s.std():>11.3f}")

feature                   raw mean   raw std |  scaled mean scaled std
x                           -1.038    41.703 |       -0.205      1.135
future_known_feature        -0.284     0.641 |       -0.071      0.160
static_feature               0.382     0.325 |       -0.044      0.491


Lets see one of the scalers

In [31]:
adapter = dm_scaled._scalers["x"]
print(type(adapter._scaler).__name__)
print("mean:", adapter._scaler.mean_, "scale:", adapter._scaler.scale_)

StandardScaler
mean: [6.50502584] scale: [36.75455738]


We can also normalize the targets - by passing `target_normalizer` to `EncoderDecoderTimeSeriesDataModule`:

- By default, no normalization
- Can pass `"auto"` to automatically choose a `target_normalizer` which best suites your data
- Or Pass the target_normalizer explicitly
  - Supported target_normalizers
      - `pytorch_forecasting.data.encoders.TorchNormalizer`,
      - `pytorch_forecasting.data.encoders.GroupNormalizer`,
      - `pytorch_forecasting.data.encoders.NaNLabelEncoder`,
      - `pytorch_forecasting.data.encoders.EncoderNormalizer`
- In case of multi-target, pass a list of normalizers for each target

`"auto"` selection of `target_normalizer`.

In [82]:
dm = EncoderDecoderTimeSeriesDataModule(
    time_series_dataset=dataset,
    max_encoder_length=30,
    max_prediction_length=12,
    target_normalizer="auto",
)

In [83]:
dm.setup(stage="fit")
adapter = dm._target_normalizer
scaler = adapter._scaler
name = type(scaler).__name__ if scaler is not None else "None (no normalization)"
print(f"{name:<20} ")


EncoderNormalizer    


We will look at using other models in the exercise sheet along with trying out different configs for data modules

## Some things to keep in mind while trying out things in v2
- We still have no categorical support
  - The pipeline assume everything is numeric!
- Limited support for `DistributionLoss`
- Train-test splitting is not implemented extensively
  - Only splitting based on groups
  - Eg, first 2 groups in train, 1 in test
- Not ALL v1 models are implemented in v2
- No hyperparam optimization